# LLM Classification Finetuning — v0.2 Llama-3.1-8B LoRA

**Approach:** Llama-3.1-8B-Instruct fine-tuned with LoRA on Kaggle GPU. Label-token readout at inference (softmax over A/B/tie logits) → 3-class probabilities for log loss.

**Strategy:**
- 5-fold stratified split; train on 4 folds, eval on held-out fold
- A/B swap augmentation to fight position bias
- Swap-TTA at inference: average (A,B) + (B,A) predictions
- GPU: P100/T4 | Internet: on

**Model source:** Loaded from Kaggle Models (`meta-llama/llama-3.1-8b-instruct`) — no HF token required.

**Runtime budget:** ~9 h cap. Train 1 fold only for first run to validate pipeline.
Scale to all 5 folds once timing is confirmed.

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.44', 'peft>=0.12', 'trl>=0.10',
    'accelerate>=0.33', 'bitsandbytes>=0.43', 'datasets>=2.20',
], check=True)

print('Dependencies installed')

In [ ]:
# ── Imports & config ──────────────────────────────────────────────────────────
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

OUTPUT = Path('/kaggle/working')

# Locate competition data
_candidates = [Path('/kaggle/input/llm-classification-finetuning')]
INPUT = next((p for p in _candidates if (p / 'train.csv').exists()), None)
if INPUT is None:
    _found = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if _found:
        INPUT = Path(_found[0]).parent

print(f'Input path : {INPUT}')
print(f'GPU        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'GPU count  : {torch.cuda.device_count()}')

# ── Hyperparameters ───────────────────────────────────────────────────────────
MODEL_NAME   = 'meta-llama/Llama-3.1-8B-Instruct'
N_FOLDS      = 5
TRAIN_FOLD   = 0          # fold index to train (0 = first validation run)
MAX_SEQ_LEN  = 1024       # tokens; increase if runtime allows
BATCH_SIZE   = 1
GRAD_ACCUM   = 16
NUM_EPOCHS   = 1
LR           = 2e-4
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
SEED         = 42

LABEL_TOKENS = ['A', 'B', 'tie']  # label-token readout targets

In [ ]:
# ── Imports & config ──────────────────────────────────────────────────────────
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

OUTPUT = Path('/kaggle/working')

# Locate competition data
_candidates = [Path('/kaggle/input/llm-classification-finetuning')]
INPUT = next((p for p in _candidates if (p / 'train.csv').exists()), None)
if INPUT is None:
    _found = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if _found:
        INPUT = Path(_found[0]).parent

print(f'Input path : {INPUT}')
print(f'GPU        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'GPU count  : {torch.cuda.device_count()}')

# ── Hyperparameters ───────────────────────────────────────────────────────────
# Model loaded from Kaggle Models (no HF auth required)
MODEL_NAME   = '/kaggle/input/llama-3.1-8b-instruct/transformers/8b-instruct/1'
N_FOLDS      = 5
TRAIN_FOLD   = 0          # fold index to train (0 = first validation run)
MAX_SEQ_LEN  = 1024       # tokens; increase if runtime allows
BATCH_SIZE   = 1
GRAD_ACCUM   = 16
NUM_EPOCHS   = 1
LR           = 2e-4
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
SEED         = 42

LABEL_TOKENS = ['A', 'B', 'tie']  # label-token readout targets
print(f'Model path : {MODEL_NAME}')

In [ ]:
# ── Prompt builder ────────────────────────────────────────────────────────────
SYSTEM = (
    'You are an expert judge evaluating AI assistant responses. '
    'Given a user prompt and two responses (A and B), output which response '
    'a human would prefer: A, B, or tie.'
)

def build_prompt(row, swap=False):
    ra = str(row['response_a'] or '')
    rb = str(row['response_b'] or '')
    if swap:
        ra, rb = rb, ra
    prompt = str(row['prompt'] or '')
    max_resp_chars = MAX_SEQ_LEN * 3
    user_msg = (
        f'PROMPT:\n{prompt}\n\n'
        f'RESPONSE A:\n{ra[:max_resp_chars]}\n\n'
        f'RESPONSE B:\n{rb[:max_resp_chars]}\n\n'
        f'Which response is better? Answer with only: A, B, or tie.'
    )
    return user_msg

def label_for_swap(label_str, swap):
    if not swap or label_str == 'tie':
        return label_str
    return 'B' if label_str == 'A' else 'A'

print('Prompt builder ready')

In [ ]:
# ── Tokenizer + fold split + swap-augmented training set ─────────────────────
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def make_chat_text(row, swap=False):
    messages = [
        {'role': 'system',    'content': SYSTEM},
        {'role': 'user',      'content': build_prompt(row, swap)},
        {'role': 'assistant', 'content': label_for_swap(row['label_str'], swap)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(train, train['label']))
tr_idx, val_idx = splits[TRAIN_FOLD]

df_tr  = train.iloc[tr_idx].reset_index(drop=True)
df_val = train.iloc[val_idx].reset_index(drop=True)

train_texts = []
for _, row in df_tr.iterrows():
    train_texts.append(make_chat_text(row, swap=False))
    train_texts.append(make_chat_text(row, swap=True))

train_ds = Dataset.from_dict({'text': train_texts})
print(f'Fold {TRAIN_FOLD}: train={len(df_tr)} ({len(train_texts)} w/ swap aug)  val={len(df_val)}')

In [ ]:
# ── Load model with 4-bit QLoRA ───────────────────────────────────────────────
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

output_dir = OUTPUT / f'fold{TRAIN_FOLD}-adapter'

sft_config = SFTConfig(
    output_dir=str(output_dir),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=True,
    logging_steps=25,
    save_strategy='epoch',
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    max_seq_length=MAX_SEQ_LEN,
    seed=SEED,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(output_dir))
print(f'Adapter saved → {output_dir}')

In [ ]:
# ── Label-token readout inference helper ──────────────────────────────────────
label_token_ids = [
    tokenizer.encode(t, add_special_tokens=False)[0]
    for t in LABEL_TOKENS
]
print(f'Label token ids: {dict(zip(LABEL_TOKENS, label_token_ids))}')

def predict_proba(df, batch_size=4, swap=False):
    model.eval()
    all_probs = []
    rows = [r for _, r in df.iterrows()]
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        texts = []
        for row in batch:
            messages = [
                {'role': 'system', 'content': SYSTEM},
                {'role': 'user',   'content': build_prompt(row, swap=swap)},
            ]
            texts.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            ))
        enc = tokenizer(
            texts, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_SEQ_LEN
        ).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits
        last_logits  = logits[:, -1, :]
        label_logits = last_logits[:, label_token_ids]
        probs = torch.softmax(label_logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
        if (i // batch_size) % 50 == 0:
            print(f'  {i+len(batch)}/{len(rows)}')
    return np.vstack(all_probs)

print('Inference helper ready')

In [ ]:
# ── OOF evaluation ────────────────────────────────────────────────────────────
print('OOF inference (swap-TTA) …')
probs_fwd = predict_proba(df_val, swap=False)
probs_swp = predict_proba(df_val, swap=True)
probs_swp_realigned = probs_swp[:, [1, 0, 2]]
oof_probs = (probs_fwd + probs_swp_realigned) / 2

y_val = df_val['label'].values
oof_ll = log_loss(y_val, oof_probs)
print(f'Fold {TRAIN_FOLD} OOF log loss (swap-TTA): {oof_ll:.4f}')

In [ ]:
# ── Test inference & submission ────────────────────────────────────────────────
print('Test inference (swap-TTA) …')
test_fwd = predict_proba(test, swap=False)
test_swp = predict_proba(test, swap=True)
test_probs = (test_fwd + test_swp[:, [1, 0, 2]]) / 2

sub = pd.DataFrame({
    'id':             test['id'],
    'winner_model_a': test_probs[:, 0],
    'winner_model_b': test_probs[:, 1],
    'winner_tie':     test_probs[:, 2],
})
sub.to_csv(OUTPUT / 'submission.csv', index=False)

print(f'submission.csv written — {len(sub):,} rows')
print(sub.head())
print(f'Prob sums: {test_probs.sum(axis=1)[:5].round(4)}')
print(f'OOF fold {TRAIN_FOLD}: {oof_ll:.4f}  (v0.1 baseline: 1.0404)')